In [1]:
!pip install osmnx --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# Pull building footprints for the full AOI (negative_sampling_area_v2)
# ============================================================

import geopandas as gpd
import osmnx as ox

# ---- Load your AOI boundary (must be in EPSG:4326 for osmnx) ----
aoi = gpd.read_file('/kaggle/input/datasets/zeinebsinini/negative-sampling-area-v2-wgs84/negative_sampling_area_v2_wgs84.shp')  # adjust path to wherever you have it

# ---- Sanity check the AOI itself before querying ----
print("AOI CRS:", aoi.crs)
print("AOI bounds (minx, miny, maxx, maxy):", aoi.total_bounds)
# For Sidi Bou Said, bounds should be roughly:
#   minx ~10.34-10.35, maxx ~10.35-10.36
#   miny ~36.86-36.87, maxy ~36.87-36.88
# If these numbers look much wider than that, the wrong file was loaded.

aoi_polygon = aoi.geometry.unary_union  # merge into one shape if multipart

# ---- Pull all building footprints strictly within the AOI polygon ----
buildings = ox.features_from_polygon(aoi_polygon, tags={"building": True})
buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])]

# ---- Extra safety: clip again explicitly, in case osmnx returned anything
# whose bounding box merely intersects the AOI rather than sitting inside it ----
buildings = gpd.clip(buildings, aoi_polygon)

print(f"Pulled {len(buildings)} building footprints (after clipping)")
print("Buildings bounds:", buildings.total_bounds)

# ---- Save ----
buildings.to_file('aoi_buildings_clipped.geojson', driver='GeoJSON')
print("Saved aoi_buildings_clipped.geojson")

AOI CRS: EPSG:4326
AOI bounds (minx, miny, maxx, maxy): [10.33178617 36.85730197 10.3548094  36.87786182]


/tmp/ipykernel_58/1323176362.py:19: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_polygon = aoi.geometry.unary_union  # merge into one shape if multipart


Pulled 554 building footprints (after clipping)
Buildings bounds: [10.3349046 36.8585437 10.3527791 36.876016 ]
Saved aoi_buildings_clipped.geojson
